In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("ETL_Pipeline_Project") \
    .getOrCreate()

spark

# Extract - from Parquet File

In [4]:
df = spark.read.parquet("/content/sample_data/yellow_tripdata_2025-01.parquet")

df.printSchema()
df.show(5)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+------

In [5]:
df.count() # count of total records

3475226

In [7]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [8]:
# Transform — Data Cleaning

# Transform - Data Cleaning

In [9]:
# Remove invalid or missing values.
df_clean = df.dropna(subset=[
    "passenger_count",
    "trip_distance",
    "fare_amount"
])

In [10]:
# Filter Invalid Trips
df_filtered = df_clean.filter(
    (df_clean.trip_distance > 0) &
    (df_clean.fare_amount > 0)
)

In [12]:
from pyspark.sql.functions import unix_timestamp, col
# Create trip duration and average speed.
df_transformed = df_filtered.withColumn(
    "trip_duration_minutes",
    (unix_timestamp("tpep_dropoff_datetime") -
     unix_timestamp("tpep_pickup_datetime")) / 60
)

In [13]:
# Average Speed
df_transformed = df_transformed.withColumn(
    "avg_speed_kmh",
    (col("trip_distance") / col("trip_duration_minutes")) * 60
)

In [14]:
# Additional Feature (Hour of Day)
from pyspark.sql.functions import hour

df_transformed = df_transformed.withColumn(
    "pickup_hour",
    hour("tpep_pickup_datetime")
)

In [15]:
# Data Aggregation
trip_summary = df_transformed.groupBy("pickup_hour").agg(
    {"trip_distance": "avg",
     "fare_amount": "avg"}
)

trip_summary.show()

+-----------+------------------+------------------+
|pickup_hour|  avg(fare_amount)|avg(trip_distance)|
+-----------+------------------+------------------+
|         12| 22.95872355181213|2.9509529543792223|
|         22|18.510513133826553| 3.517796538623919|
|          1| 17.15054727272728|3.2473456818181843|
|         13|18.041002548599746| 3.242176277134791|
|          6|  21.5842519259743| 5.832847030730318|
|         16| 18.79064142154571|3.2630350811525117|
|          3|17.229864745815444|3.2923796445043023|
|         20| 17.74468030240148| 3.318775488383216|
|          5|27.040971432264655|6.0348442347466325|
|         19| 17.04257006264435|3.0575472674147397|
|         15|18.726462329236163| 3.224370661779452|
|          9|17.113404095054754| 3.231641342555826|
|         17|  17.3580215025945|2.9611774682707415|
|          4| 23.15145885387284| 4.785629886354475|
|          8|16.992928279033297|2.9304846819566848|
|         23| 19.64055633365297| 3.929985888943892|
|          7

# Load - Save into a table

In [ ]:
# Create Spark SQL Table
spark.sql("CREATE DATABASE IF NOT EXISTS taxi_db")

df_transformed.write \
    .mode("overwrite") \
    .saveAsTable("taxi_db.yellow_taxi_trips")

# Query the Table
spark.sql("""
SELECT pickup_hour,
       COUNT(*) as trips,
       AVG(fare_amount) as avg_fare
FROM taxi_db.yellow_taxi_trips
GROUP BY pickup_hour
ORDER BY pickup_hour
""").show()